# Export: test example

Shows what `artifact_links/scripts_artifact_links/export.py` produces for a small mixed selection: genomes from the three databases, plus two that fail.
The result is one JSON file for the whole selection, written next to this notebook (`example_export.json`).

In [1]:
import sys
sys.path.append("../..")

import json
from artifact_links.scripts_artifact_links.export import collect_links, write_export

## 1. The selection

Each genome says which database it comes from and its accession.
The second one also gives the extra information that Colin's search results will have (assembly name and species),
so NCBI's API is not asked again for it. The last two are there on purpose, to show what happens when something fails.

In [2]:
genomes = [
    {"source": "NCBI", "accession": "GCF_018350175.1"},
    {"source": "NCBI", "accession": "GCA_018350175.1", "assembly_name": "F.catus_Fca126_mat1.0", "species": "Felis catus"},
    {"source": "GWH", "accession": "GWHBHRQ00000000"},
    {"source": "GWH", "accession": "GWHBJIX01000000"},
    {"source": "Ensembl", "accession": "GCA_018350175.1"},
    {"source": "NCBI", "accession": "GCF_999999999.1"},
    {"source": "UCSC", "accession": "felCat9"},
]

## 2. Collect the links

There is a short pause between two genomes, so NCBI doesn't receive too many requests at once.

In [3]:
export = collect_links(genomes)

print(len(export["genomes"]), "genomes with links,", len(export["problems"]), "problems")
for genome in export["genomes"]:
    found = sum(link is not None for link in genome["links"].values())
    print(f'{genome["source"]:8} {genome["accession"]:16} {genome["species"]:22} {found} files')

5 genomes with links, 2 problems
NCBI     GCF_018350175.1  Felis catus            11 files
NCBI     GCA_018350175.1  Felis catus            4 files
GWH      GWHBHRQ00000000  Rhinopithecus bieti    7 files
GWH      GWHBJIX01000000  Canis lupus familiaris 2 files
Ensembl  GCA_018350175.1  Felis catus            6 files


## 3. The problems

The genomes that failed are not lost: each one is listed with the reason.

In [4]:
for problem in export["problems"]:
    print(problem["source"], problem["accession"], "->", problem["message"])

NCBI GCF_999999999.1 -> GCF_999999999.1 was not found on NCBI
UCSC felCat9 -> unknown database 'UCSC' (expected NCBI, GWH or Ensembl)


## 4. Write the JSON file

This is the file a researcher or a pipeline would use.

In [5]:
write_export(export, "example_export.json")
print(json.dumps(export, indent=2)[:3000])  # the beginning of the file

{
  "format": "genomexplore-artifact-links",
  "format_version": "1.0",
  "created": "2026-11-09T12:30:54",
  "genomes": [
    {
      "source": "NCBI",
      "accession": "GCF_018350175.1",
      "species": "Felis catus",
      "links": {
        "genome_fasta": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.catus_Fca126_mat1.0_genomic.fna.gz",
        "gff": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.catus_Fca126_mat1.0_genomic.gff.gz",
        "gtf": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.catus_Fca126_mat1.0_genomic.gtf.gz",
        "cds": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.catus_Fca126_mat1.0_cds_from_genomic.fna.gz",
        "rna": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_0